# 🧑‍🏫 Regression Microprojects — Instructor Notebook
*Elements of Data Science* <br>
This notebook does three jobs, for **all of your classes at once**:

1. **Reads each class roster** exactly as exported and randomly assigns every student to one of five microprojects, dealt separately within each class
2. **Saves the assignments** to `/home/jovyan/shared-public/microprojects/assignments.csv`, where each student's `my_project(name)` finds them by hub login
3. Shows the live **gallery** of submitted figures and who has submitted

It is self-contained: the roster reading is done in this notebook, so you can see and edit it.

**Before you start:** upload one roster per class into **this folder on the hub**, named `class_<section number>.csv`, e.g. `class_01.csv` and `class_08.csv`. Files with other names (such as `class_list.csv` or `class_sample.csv`) are ignored. 🔒 Never commit rosters to GitHub: they hold student names and IDs.

## ⚙️ Setup

In [ ]:
import sys, os, glob, json, csv, re, hashlib
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "shared-public")))   # Fall 2026/shared-public
try:
    from init import *
except ModuleNotFoundError:
    print('init.py not found from this folder; continuing without the course styling.')

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
# Where students' notebooks look for assignments and put submissions — must match microproject_tools.py
MICRO_ROOT  = os.environ.get('MICRO_ROOT', '/home/jovyan/shared-public/microprojects')
SUBMIT_DIR  = os.path.join(MICRO_ROOT, 'submissions')
ASSIGN_FILE = os.path.join(MICRO_ROOT, 'assignments.csv')
os.makedirs(SUBMIT_DIR, exist_ok=True)
for folder in [MICRO_ROOT, SUBMIT_DIR]:
    try:
        os.chmod(folder, 0o777)          # every student must be able to write a submission
    except PermissionError:
        pass

projects = Table().with_columns(
    'project', make_array('MP1', 'MP2', 'MP3', 'MP4', 'MP5'),
    'title',   make_array('Carbon Dioxide at Mauna Loa', "Galton's Heights", 'Birth Weight and Gestation',
                          'Palmer Penguins', 'Periodic Trends'))

def user_key(login):
    "Scrambled stand-in for a hub login (same formula as microproject_tools.py), so IDs never go in the shared file."
    login = str(login).strip().lower()
    return hashlib.sha256(('eds-microprojects:' + login).encode()).hexdigest()[:12] if login else ''

print('Assignments file :', ASSIGN_FILE)
print('Submissions      :', SUBMIT_DIR)
projects

## 1. Read the Class Rosters
Each roster is read in the layout it was exported in. Two layouts are handled:

|Layout|First line|
|---|---|
|Registrar export, **no header**|`"Ahmed, Anas",tut13755,8,,,,`|
|Canvas export, **with header**|`Student,SIS Login ID,Section,,,,`|

Either way, the name is `"Last, First Middle"` and we keep the **first given name** and the **last name**. Canvas's *Test Student* (and any *Points Possible* row) is skipped. The class label comes from the file name: `class_01.csv` → `01`.

In [ ]:
def read_roster(path):
    "One class roster -> list of students, each a dict with section, first, last, login."
    with open(path, newline='', encoding='utf-8-sig') as f:
        rows = [[cell.strip() for cell in row] for row in csv.reader(f)]
    rows = [row for row in rows if any(row)]                     # drop blank lines

    # Header row? (Canvas: 'Student, SIS Login ID, Section')  Otherwise names are column 0, logins column 1.
    header = [cell.lower() for cell in rows[0]]
    name_col, login_col = 0, 1
    if 'student' in header or 'name' in header:
        name_col = header.index('student') if 'student' in header else header.index('name')
        for label in ['sis login id', 'login id', 'login', 'username', 'user']:
            if label in header:
                login_col = header.index(label)
                break
        rows = rows[1:]

    section = re.findall(r'\d+', os.path.basename(path))[-1].zfill(2)
    students = []
    for row in rows:
        full = row[name_col]
        if full.lower().startswith(('student, test', 'test student')) or 'points possible' in full.lower():
            continue                                             # Canvas test student / points row
        if ',' in full:
            last, given = full.split(',', 1)                     # 'Last, First Middle'
        else:
            given, last = full.rsplit(' ', 1) if ' ' in full else (full, '')
        given_names = given.split()
        if not given_names:
            continue
        login = row[login_col].split('@')[0].lower() if len(row) > login_col else ''
        students.append({'section': section, 'first': given_names[0], 'last': last.strip(), 'login': login})
    return students

class_files = sorted(glob.glob('class_[0-9]*.csv'))
print('Roster files found:', class_files)

all_students = []
for f in class_files:
    one_class = read_roster(f)
    print(f'  {f}: {len(one_class)} students, class {one_class[0]["section"]}')
    all_students = all_students + one_class

Now put everyone in one Table. Within each class, students who share a first name are shown with a last initial ("Justin C.", "Justin K.").

In [ ]:
sections = make_array(*[s['section'] for s in all_students])
firsts   = make_array(*[s['first'] for s in all_students])
lasts    = make_array(*[s['last'] for s in all_students])

display_names = make_array()
for i in np.arange(len(firsts)):
    same_class_same_first = np.count_nonzero((sections == sections.item(i)) & (firsts == firsts.item(i)))
    label = f'{firsts.item(i)} {lasts.item(i)[:1]}.' if same_class_same_first > 1 else firsts.item(i)
    display_names = np.append(display_names, label)

keys = make_array()
for s in all_students:
    # Use the login; if a roster has none, fall back to a key built from class + name
    keys = np.append(keys, user_key(s['login']) or user_key(f"{s['section']}|{s['first']}|{s['last']}"))

roster = Table().with_columns(
    'section',      sections,
    'display_name', display_names,
    'first_name',   firsts,
    'last_name',    lasts,
    'last_initial', make_array(*[l[:1].upper() for l in lasts]),
    'user_key',     keys)
roster.group('section')

👀 **Check the names** — the first few students of each class:

In [ ]:
for sec in np.unique(roster.column('section')):
    print('Class', sec)
    roster.where('section', sec).select('display_name', 'first_name', 'last_name').show(6)

## 2. Assign Projects
Within each class we shuffle the students with `np.random.choice(..., replace=False)` — a random **permutation** — and deal them to the projects like cards, so group sizes differ by at most one. The seed makes the deal reproducible.

**Saved assignments are respected.** Students who already have a project keep it. Students new to a roster are dealt into the smallest groups, and students no longer on a roster are dropped. Leftovers from earlier test runs (a class with no roster file, or names that aren't on any roster) are cleared automatically. To start a class over before students have seen it, list it in `redeal`.

In [ ]:
redeal = make_array()          # e.g. make_array('01') to re-deal class 01 from scratch
seed = 1013

# Previously saved assignments (if any) as a dict: user_key -> project
saved = {}
if os.path.exists(ASSIGN_FILE):
    old = Table.read_table(ASSIGN_FILE, dtype=str)
    if 'user_key' in old.labels and 'project' in old.labels:
        for key, pid in zip(old.column('user_key'), old.column('project')):
            saved[key] = pid
print(len(saved), 'saved assignments found')

def assign_class(class_roster, seed):
    "Keep saved projects; deal everyone else into the currently smallest groups."
    np.random.seed(seed)
    section = class_roster.column('section').item(0)
    use_saved = section not in redeal
    pids = projects.column('project')
    project_list = make_array()
    counts = {pid: 0 for pid in pids}
    for key in class_roster.column('user_key'):                       # first pass: who keeps a project
        pid = saved.get(key, '') if use_saved else ''
        project_list = np.append(project_list, pid)
        if pid:
            counts[pid] += 1
    kept = np.count_nonzero(project_list != '')
    new_students = np.random.choice(np.arange(class_roster.num_rows), class_roster.num_rows, replace=False)
    for i in new_students:                                             # second pass: deal the rest
        if project_list.item(i) == '':
            order = np.random.choice(pids, len(pids), replace=False)  # random tie-breaking
            smallest = min(order, key=lambda p: counts[p])
            project_list[i] = smallest
            counts[smallest] += 1
    print(f'Class {section}: {kept} kept their saved project, {class_roster.num_rows - kept} newly assigned')
    return class_roster.with_columns('project', project_list)

class_ids = np.unique(roster.column('section'))
assignments = assign_class(roster.where('section', class_ids.item(0)), seed)
for i in np.arange(1, len(class_ids)):
    assignments = assignments.append(assign_class(roster.where('section', class_ids.item(i)), seed + i))

assignments = assignments.select('section', 'display_name', 'first_name', 'last_initial', 'user_key', 'project')
assignments.to_csv(ASSIGN_FILE)
try:
    os.chmod(ASSIGN_FILE, 0o644)
except PermissionError:
    pass
print('Saved', assignments.num_rows, 'assignments to', ASSIGN_FILE)
assignments.pivot('project', 'section')

Each group, ready to show on screen. Set `show_class` to one class (e.g. `'08'`), or `None` for all.

In [ ]:
show_class = None

for sec in np.unique(assignments.column('section')):
    if show_class is not None and sec != show_class:
        continue
    print(f'── Class {sec} ──')
    one_class = assignments.where('section', sec)
    for pid, title in zip(projects.column('project'), projects.column('title')):
        names = sorted(one_class.where('project', pid).column('display_name'))
        print(f'  {pid} — {title:28s} ({len(names)})  {", ".join(names)}')
    print()

### 🎲 The Draw — reveal one class's groups live
Project this cell on screen. Students are revealed **one at a time in random order** onto five group Tables, with a drumroll before each name. The projects were already randomly assigned and saved above; the draw just reveals them, so rerunning it never changes anyone's group.

Set `draw_class` to the class in front of you and `pause` to the seconds per student (≈0.8 s × 58 students ≈ 1–2 minutes; use `0` to show the final board instantly).

In [ ]:
from IPython.display import display, HTML
import time

draw_class = '08'          # which class to reveal
pause = 0.8                # seconds per student

emoji = {'MP1': '🌍', 'MP2': '📏', 'MP3': '👶', 'MP4': '🐧', 'MP5': '⚛️'}
titles = dict(zip(projects.column('project'), projects.column('title')))

def board_html(revealed, message, highlight=None):
    "Five group Tables side by side, with a big message line on top."
    cards = ''
    for pid in projects.column('project'):
        names = revealed[pid]
        group = Table().with_columns(f'{len(names)} students', make_array(*names) if names else make_array())
        border = '#F2B705' if pid == highlight else '#A7A8AA'
        glow = 'box-shadow:0 0 18px #F2B705;' if pid == highlight else ''
        cards += (f'<div style="flex:1 1 180px;min-width:170px;border:3px solid {border};border-radius:12px;'
                  f'overflow:hidden;{glow}transition:all .3s">'
                  f'<div style="background:#A41E35;color:white;padding:8px 10px;font-weight:800;font-size:1.15em">'
                  f'{emoji[pid]} {pid}<div style="font-size:.75em;font-weight:600">{titles[pid]}</div></div>'
                  f'<div style="padding:4px 8px;font-size:1.05em">{group.as_html()}</div></div>')
    return (f'<div style="font-family:sans-serif">'
            f'<div style="font-size:2em;font-weight:900;margin:6px 0 12px;min-height:1.4em">{message}</div>'
            f'<div style="display:flex;flex-wrap:wrap;gap:10px;align-items:flex-start">{cards}</div></div>')

def draw(section, pause=0.8):
    "Reveal a class's saved groups one student at a time."
    one_class = assignments.where('section', section)
    n = one_class.num_rows
    revealed = {pid: [] for pid in projects.column('project')}
    board = display(HTML(board_html(revealed, f'🎲 Class {section}: {n} students, 5 projects…')), display_id=True)
    time.sleep(2 * pause)
    order = np.random.choice(np.arange(n), n, replace=False)          # random reveal order
    for k, i in enumerate(order):
        name, pid = one_class.column('display_name').item(i), one_class.column('project').item(i)
        board.update(HTML(board_html(revealed, f'🥁 Student {k + 1} of {n}…')))
        time.sleep(pause / 2)
        revealed[pid].append(name)
        board.update(HTML(board_html(revealed, f'<span style="color:#A41E35">{name}</span> → {emoji[pid]} {titles[pid]}', highlight=pid)))
        time.sleep(pause)
    board.update(HTML(board_html(revealed, f'🎉 Class {section}: all {n} students drawn!')))

draw(draw_class, pause)

Or show any class's groups instantly as Tables (no animation):

In [ ]:
for pid in projects.column('project'):
    group = assignments.where('section', draw_class).where('project', pid).select('display_name').sort('display_name')
    print(f'{emoji[pid]} {pid}: {titles[pid]} ({group.num_rows} students)')
    group.relabeled('display_name', 'student').show()

## 3. Submissions 🖼️
Rerun these cells any time during class. Each submission is a PNG plus a small JSON file with the student's *r*, slope, intercept, caption, and class.

In [ ]:
def load_submissions():
    "Read every submission record into a Table."
    paths = sorted(glob.glob(os.path.join(SUBMIT_DIR, '*.json')))
    records = []
    for p in paths:
        r = json.load(open(p))
        r['png'] = p[:-5] + '.png'
        records.append(r)
    return Table().with_columns(
        'project',   make_array(*[r.get('project', '') for r in records]),
        'section',   make_array(*[str(r.get('section', '')) for r in records]),
        'name',      make_array(*[r.get('name', '') for r in records]),
        'user_key',  make_array(*[r.get('user_key', '') for r in records]),
        'r',         make_array(*[r.get('r', np.nan) for r in records]),
        'slope',     make_array(*[r.get('slope', np.nan) for r in records]),
        'intercept', make_array(*[r.get('intercept', np.nan) for r in records]),
        'caption',   make_array(*[r.get('caption', '') for r in records]),
        'submitted', make_array(*[r.get('submitted', '') for r in records]),
        'png',       make_array(*[r['png'] for r in records]))

submissions = load_submissions()
print(submissions.num_rows, 'submissions')
if submissions.num_rows:
    submissions.drop('png', 'user_key').sort('submitted', descending=True).show(10)

### Who has submitted?

In [ ]:
show_class = None          # e.g. '08'

done = make_array(*['✅' if key in submissions.column('user_key') else '—'
                    for key in assignments.column('user_key')])
status = assignments.with_columns('submitted', done)
if show_class is not None:
    status = status.where('section', show_class)
print(np.count_nonzero(status.column('submitted') == '✅'), 'of', status.num_rows, 'students have submitted')
status.pivot('submitted', 'section').show()
status.select('section', 'project', 'display_name', 'submitted').sort('display_name').sort('section').show()

### Show the figures
Each project gets its own banner; each figure is a card with the student's name (as it appears in the groups), their caption, and their numbers. Set `show_class` above to show one class, and `card_width` to make cards larger or smaller on the projector.

In [ ]:
import base64, html
from IPython.display import display, HTML

card_width = 460           # pixels; cards wrap to fit the screen
emoji = {'MP1': '🌍', 'MP2': '📏', 'MP3': '👶', 'MP4': '🐧', 'MP5': '⚛️', 'DEMO': '🚗'}
titles = dict(zip(projects.column('project'), projects.column('title')))
titles['DEMO'] = 'Instructor demo'

# Show each student by their group name ('Justin C.'), whatever they typed in their notebook
group_name = dict(zip(assignments.column('user_key'), assignments.column('display_name')))

def card(i, sub):
    "One figure card: image, name, caption, numbers."
    with open(sub.column('png').item(i), 'rb') as f:
        img = base64.b64encode(f.read()).decode()
    key = sub.column('user_key').item(i)
    who = html.escape(group_name.get(key, sub.column('name').item(i)))
    sec = sub.column('section').item(i)
    caption = html.escape(sub.column('caption').item(i)) or '<i>(no caption)</i>'
    numbers = f"r = {sub.column('r').item(i):.3f} &nbsp;·&nbsp; slope = {sub.column('slope').item(i):.4g}"
    return (f'<div style="background:white;border:1px solid #ccc;border-radius:10px;padding:10px 12px;'
            f'box-shadow:0 2px 6px rgba(0,0,0,.15)">'
            f'<div style="font-size:1.3em;font-weight:800;margin-bottom:4px">{who}'
            f'<span style="font-size:.6em;color:#777;font-weight:600"> &nbsp;class {sec}</span></div>'
            f'<img src="data:image/png;base64,{img}" style="width:100%;border-radius:6px">'
            f'<div style="font-size:1.05em;line-height:1.35;margin-top:8px">{caption}</div>'
            f'<div style="font-size:.85em;color:#555;margin-top:6px">{numbers}</div></div>')

def show_gallery(pid, section=None):
    "A banner for one project, then every submitted figure for it as cards."
    sub = submissions.where('project', pid)
    if section is not None:
        sub = sub.where('section', section)
    banner = (f'<div style="background:#A41E35;color:white;border-radius:10px;padding:12px 18px;margin:28px 0 12px;'
              f'font-size:1.6em;font-weight:900">{emoji.get(pid, "")} {pid}: {titles.get(pid, pid)}'
              f'<span style="font-size:.6em;font-weight:600"> &nbsp;— {sub.num_rows} figure{"s" if sub.num_rows != 1 else ""}'
              f'{" · class " + section if section else ""}</span></div>')
    if sub.num_rows == 0:
        display(HTML(banner + '<div style="color:#777;margin-left:6px">No submissions yet.</div>'))
        return
    cards = ''.join(card(i, sub) for i in np.arange(sub.num_rows))
    grid = (f'<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax({card_width}px,1fr));'
            f'gap:16px;align-items:start">{cards}</div>')
    display(HTML(banner + grid))

for pid in projects.column('project'):
    show_gallery(pid, section=show_class)

## 4. Discussion: Did Everyone Get the Same Line?
Students on the same project used the same data, so their full-data *r* and slope **should agree**. Disagreement is a good discussion starter: a toolbox bug, a subset of the data (the twist line), or swapped $x$ and $y$ ([Ch. 15.2](https://inferentialthinking.com/chapters/15/2/Regression_Line.html)).

In [ ]:
if submissions.num_rows:
    submissions.select('project', 'r', 'slope').group('project', np.mean).show()

**Gallery-walk prompts**
- Which figure would you put in a newspaper? Why?
- Which project shows that a large *r* does not guarantee a straight line is the right model?
- In which projects did splitting the data into groups change the story?
- Which findings could you state as **causes**, and which only as **associations**?